# Kikuchi Maps: Planning The Route To The Next Zone Axis

You are sitting at the microscope with the beam down $[001]$ of a nickel grain, and you want
$[111]$. The screen shows perhaps six bands and three or four of their intersections. Which way do
you turn the goniometer, how far, and how will you know you are still on course halfway there?

The classical answer is a **Kikuchi map**: the whole band network of the crystal, drawn once on a
stereographic projection, so the route is read off a picture instead of guessed. Levine, Bell and
Thomas built the first ones in 1966 by montaging dozens of exposures taken at known tilts, and a
good map was a day's work. It was worth a day's work, because with the map in front of you the
navigation problem becomes trivial: **find a band that passes through both zone axes and follow
it.**

This tutorial computes such maps from the lattice, for a cubic and a hexagonal phase, and turns
them into routes. Along the way three things fall out that are worth knowing independently of the
microscope:

- band width goes as $1/d$, so the *widest* bands are the *highest-index* ones while the strongest
  are the lowest — two orderings that are easy to conflate;
- the projection has to be stereographic, and the reason is quantitative: at $89.99^\circ$ from the
  map centre the gnomonic radius is 5730 and the stereographic radius is 0.99983;
- following a band *is* taking the geodesic, and the penalty for taking one long hop instead of
  several short ones is an exact expression, $2\arcsin(\sin(\delta\varphi/2)\sin\theta)$, that you
  can evaluate before deciding.

## Learning goals

1. What is a Kikuchi band on the *sphere*, and why does its width measure $d$ inversely?
2. Why must the atlas be stereographic when the detector pattern is gnomonic?
3. What does the Weiss zone law look like as arithmetic, and why does the number of bands at an
   axis matter more than the axis's indices?
4. Why is following a band the same as taking the shortest path, and when is a long tilt worse than
   several short ones?
5. What does the same construction look like for a hexagonal crystal, and what changes?

## 0. Setup

Two phases from the pinned CIF fixtures: nickel for the cubic case and zirconium for the hexagonal
one. Both maps are computed at 200 kV.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF")
warnings.filterwarnings("ignore", message="No _symmetry_equiv_pos_as_xyz")

from pytex import (
    FrameDomain,
    ReferenceFrame,
    compute_kikuchi_map,
    get_phase_fixture,
    plan_kikuchi_route,
    plot_kikuchi_map,
)

np.set_printoptions(precision=4, suppress=True)

CRYSTAL = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"))
NICKEL = get_phase_fixture("ni_fcc").load_phase(crystal_frame=CRYSTAL)
ZIRCONIUM = get_phase_fixture("zr_hcp").load_phase(crystal_frame=CRYSTAL)
BEAM_KEV = 200.0

for phase in (NICKEL, ZIRCONIUM):
    lattice = phase.lattice
    print(f"{phase.name:<16} {phase.symmetry.to_point_group().hermann_mauguin:<7} "
          f"a = {lattice.a:.4f} A, c = {lattice.c:.4f} A")

## 1. A band is a pair of cones, and its width measures the lattice

Electrons scatter incoherently inside the foil, so they travel in every direction. For a lattice
plane of spacing $d$, those leaving at the Bragg angle $\theta_B$ to the plane diffract, and the
diffracting directions form two cones of semi-angle $90^\circ - \theta_B$ about the plane normal
$\mathbf{g}$ — the **Kossel cones**. On the unit sphere of directions that is a band:

- the **centre line** is the great circle perpendicular to $\mathbf{g}$: the *trace* of the plane;
- the **edges** are the two small circles at $90^\circ \mp \theta_B$ from $\mathbf{g}$;
- the **width** is exactly $2\theta_B$, with $\sin\theta_B = \lambda/2d$.

So a band width is a direct measurement of an interplanar spacing. Note which way round:

$$2\theta_B = 2\arcsin\!\left(\frac{\lambda}{2d}\right) \approx \frac{\lambda}{d},$$

which *falls* as $d$ *rises*. The bands you can measure most precisely are the narrow ones from
low-index planes, and the fat conspicuous ones come from high-index planes.

In [ ]:
CUBIC_MAP = compute_kikuchi_map(
    NICKEL, beam_energy_kev=BEAM_KEV, max_index=4, zone_axis_max_index=4,
    min_zone_axis_order=3,
)
print(CUBIC_MAP.describe())

In [ ]:
wavelength = CUBIC_MAP.wavelength_angstrom
print(f"electron wavelength at {BEAM_KEV:.0f} kV: {wavelength:.5f} A\n")
print(f"{'plane':<12} {'d (A)':>8} {'theta_B (deg)':>14} {'width (deg)':>12} {'intensity':>10}")
for band in CUBIC_MAP.bands[:8]:
    expected = 2.0 * np.degrees(np.arcsin(wavelength / (2.0 * band.d_spacing_angstrom)))
    assert abs(band.angular_width_deg - expected) < 1e-12
    print(f"{str(band.indices):<12} {band.d_spacing_angstrom:>8.4f} {band.bragg_angle_deg:>14.4f}"
          f" {band.angular_width_deg:>12.4f} {band.relative_intensity:>10.4f}")

widest = max(CUBIC_MAP.bands, key=lambda band: band.angular_width_deg)
finest = min(CUBIC_MAP.bands, key=lambda band: band.d_spacing_angstrom)
print(f"\nwidest band  : {widest.indices}  d = {widest.d_spacing_angstrom:.4f} A, "
      f"width {widest.angular_width_deg:.3f} deg, intensity {widest.relative_intensity:.4f}")
print(f"strongest    : {CUBIC_MAP.bands[0].indices}  "
      f"d = {CUBIC_MAP.bands[0].d_spacing_angstrom:.4f} A, "
      f"width {CUBIC_MAP.bands[0].angular_width_deg:.3f} deg, intensity 1.0000")
assert widest.indices == finest.indices

The strongest band on a nickel map is $\{111\}$ — the close-packed plane, the largest spacing, and
therefore the *narrowest* of the low-index set at $0.706^\circ$. The widest is $\{442\}$ at
$2.447^\circ$, three and a half times as wide and carrying five per cent of the intensity. That is
the conflation the table exists to prevent: **width tells you $d$; intensity tells you whether you
will see it at all.**

> **The awe note.** Put a number on the width. At 200 kV the electron wavelength is 0.02508 Å, so
> for a 2 Å spacing $2\theta_B$ is $0.72^\circ$. The bands of a TEM Kikuchi pattern are *fractions
> of a degree wide*, spread across a field of view tens of degrees across. A Kikuchi map is
> therefore a network of essentially infinitely thin lines — which is exactly why it works as a
> map. If the bands were degrees wide they would overlap into a wash and the intersections, which
> are the landmarks, would not be locatable. The narrowness of the band is not an inconvenience; it
> is the property that makes navigation possible.

The figure below sweeps the relation and marks the low-index nickel planes on it.

In [ ]:
spacings = np.linspace(0.5, 3.6, 400)
widths = 2.0 * np.degrees(np.arcsin(wavelength / (2.0 * spacings)))

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.1))
axes[0].plot(spacings, widths, color="#1f77b4", label=r"$2\arcsin(\lambda/2d)$")
axes[0].plot(spacings, np.degrees(wavelength / spacings), "--", color="0.5",
             label=r"small-angle $\lambda/d$")
for band in CUBIC_MAP.bands[:6]:
    axes[0].plot([band.d_spacing_angstrom], [band.angular_width_deg], "o", ms=5, color="#d62728")
    axes[0].annotate(str(band.indices), (band.d_spacing_angstrom, band.angular_width_deg),
                     textcoords="offset points", xytext=(5, 3), fontsize=7.5)
axes[0].set_xlabel(r"$d$ (Å)"), axes[0].set_ylabel(r"band width $2\theta_B$ (deg)")
axes[0].set_title("width measures the spacing, inversely")
axes[0].legend(fontsize=8)

intensity = np.array([band.relative_intensity for band in CUBIC_MAP.bands])
width = np.array([band.angular_width_deg for band in CUBIC_MAP.bands])
axes[1].scatter(width, intensity, s=18, color="#2ca02c", alpha=0.75)
axes[1].set_xlabel("band width (deg)"), axes[1].set_ylabel("relative kinematic intensity")
axes[1].set_yscale("log")
axes[1].set_title("wide and strong are different orderings")
fig.tight_layout()

The right panel is the whole point in one scatter: there is no positive relation between width and
intensity at all. If anything it is negative, because a large spacing gives both a strong
reflection and a narrow band.

## 2. Why the atlas is stereographic and the pattern is gnomonic

`pytex.diffraction.kikuchi` draws bands in **gnomonic** coordinates, and does so for a good
reason: gnomonic projection is central projection onto a plane, so great circles map to *straight
lines*, and band centre lines on a flat detector are therefore exactly straight whatever the
detector tilt. For reading a physical pattern that is the right frame.

It cannot be the frame for an atlas, and the reason is not aesthetic. A gnomonic projection maps a
direction at polar angle $\psi$ to radius $\tan\psi$, which diverges at $90^\circ$: the projection
of a *hemisphere* is the whole infinite plane. A map has to show the hemisphere.

The stereographic projection maps $\psi$ to $\tan(\psi/2)$, so the hemisphere becomes the unit
disc; it is conformal, so angles measured on the map are true angles; and great circles become
circular arcs. Since every number an operator reads off the map is an angle the stage must turn
through, conformality is the property that matters.

In [ ]:
print(f"{'polar (deg)':>12} {'gnomonic':>12} {'stereographic':>15} {'equal-area':>12}")
for polar_deg in (10.0, 45.0, 80.0, 89.0, 89.9, 89.99):
    polar = np.radians(polar_deg)
    print(f"{polar_deg:>12} {np.tan(polar):>12.2f} {np.tan(polar / 2.0):>15.5f}"
          f" {2.0 * np.sin(polar / 2.0):>12.5f}")
print("\nThe gnomonic radius of the map boundary is infinite; the stereographic radius is 1.")
assert np.isclose(np.tan(np.radians(90.0) / 2.0), 1.0)

At $89.99^\circ$ — a fifth of a milliradian from the boundary — the gnomonic radius is 5730 times
the projection distance while the stereographic radius is 0.99983. The two projections are not
alternatives on the same footing; one of them simply cannot draw the picture.

The equal-area (Schmidt) column is there because it is the other common choice, and it is the wrong
one *here* for a stated reason: it preserves area, not angle, so a $10^\circ$ tilt near the rim and
a $10^\circ$ tilt near the centre would subtend different angles on the plot. For a texture pole
figure, where the quantity of interest is a density per unit solid angle, that trade is the right
way round. For navigation it is not. `plot_kikuchi_map` defaults to stereographic and accepts
`method="equal_area"`.

## 3. The zone law, as arithmetic

A direction $[uvw]$ lies on the centre line of $(hkl)$ exactly when

$$hu + kv + lw = 0,$$

the **Weiss zone law** — because the centre line is the set of directions perpendicular to
$\mathbf{g}$, and that dot product is the perpendicularity condition in the dual bases. It is an
*integer* condition, which is why a Kikuchi map is exact combinatorics rather than a numerical
coincidence: the zone axis at an intersection is $\mathbf{u} \propto \mathbf{g}_1 \times
\mathbf{g}_2$, and the bands through it are all the reflections satisfying the law.

The number of them — the axis's **order** — is what an operator actually uses. A four-band
intersection is unmistakable on the screen; a two-band one is a guess. So the map sorts its axes by
order, not by index.

> **Algorithm — building the map** (`compute_kikuchi_map`)
>
> 1. Enumerate integer triples to `max_index`, one per antipodal pair, and drop those the lattice
>    centring forbids.
> 2. Compute $\mathbf{g}$, hence $d = 1/|\mathbf{g}|$ and $\sin\theta_B = \lambda/2d$; keep the
>    reflections above the intensity and spacing thresholds.
> 3. Collapse each family of collinear triples onto its lowest allowed order, because all orders of
>    one reflection share a single centre line. The representative need not be coprime: in fcc the
>    $\{100\}$ trace is drawn by $(200)$, since $(100)$ is extinguished.
> 4. Rotate the normals into the map frame, whose $+z$ is the chosen centre direction.
> 5. Cross every pair of normals, rationalize each crossing to a low-index $[uvw]$, merge
>    duplicates, and record every band satisfying the zone law for it.

In [ ]:
print(f"{'zone axis':<12} {'order':>6} {'polar (deg)':>12}   bands through it")
for axis in CUBIC_MAP.zone_axes[:8]:
    members = [CUBIC_MAP.bands[position].indices for position in axis.band_indices]
    print(f"{str(axis.indices):<12} {axis.order:>6} {axis.polar_angle_deg:>12.2f}   "
          f"{members[:4]}{' ...' if len(members) > 4 else ''}")

violations = 0
for axis in CUBIC_MAP.zone_axes:
    direction = np.asarray(axis.indices, dtype=np.int64)
    for position in axis.band_indices:
        plane = np.asarray(CUBIC_MAP.bands[position].indices, dtype=np.int64)
        violations += int(plane @ direction) != 0
print(f"\nzone-law violations over {CUBIC_MAP.zone_axis_count} axes: {violations}")
assert violations == 0

centre_bands = CUBIC_MAP.bands_through([0, 0, 1])
print(f"bands through the map centre [001]: {len(centre_bands)}, "
      f"all with l = 0: {all(band.indices[2] == 0 for band in centre_bands)}")

Not one violation, because the condition is integer arithmetic rather than a tolerance. And the
bands through $[001]$ all have $l = 0$: the zone law for $[001]$ *is* the statement $l = 0$, so the
traces crossing the centre of the map are the planes whose normals lie in the plane of the
projection.

Here is the map. Compare it with any published standard $[001]$ cubic projection: the $\{100\}$ and
$\{110\}$ traces, the $\langle 110 \rangle$ poles on the rim, the $\langle 111 \rangle$ poles at
$54.7^\circ$.

In [ ]:
figure = plot_kikuchi_map(
    CUBIC_MAP,
    max_bands=18,
    width_scale=8.0,
    min_label_order=8,
    max_labels=12,
    title="Nickel Kikuchi map, centred on [001] (band widths x8 for legibility)",
)

Band widths are drawn eight times their true value, which the generated title says whenever
`width_scale` is not 1. At true scale on a printed page the bands would be single-pixel lines: that
is what section 1 measured, not a defect of the renderer.

## 4. Routing: the band *is* the geodesic

Two zone axes are joined by a band exactly when one reflection is perpendicular to both — that is,
when the plane the two directions span is a rational lattice plane. And the shortest arc on a
sphere between two directions lies in the plane they span. So:

**The great circle an operator follows by eye, tracking a band, is the geodesic.** The optimal path
and the followable path are the same path, which is not a coincidence to be admired but a
consequence of both being defined by the plane the two axes span.

For $[001]$ and $[111]$ the spanning plane has normal $[001] \times [111] \propto (1\bar{1}0)$, so
the $\{110\}$ band runs between them, and the arc length is the closed-form angle
$\arccos(1/\sqrt3) = 54.7356^\circ$.

In [ ]:
shared = CUBIC_MAP.shared_bands([0, 0, 1], [1, 1, 1])
print("bands joining [001] and [111]:", [band.indices for band in shared])
print("each has h + k + l = 0:", all(sum(band.indices) == 0 for band in shared))

single = CUBIC_MAP.route_to([0, 0, 1], [1, 1, 1], max_leg_deg=60.0)
exact = np.degrees(np.arccos(1.0 / np.sqrt(3.0)))
print(f"\nsingle-hop route: {single.hop_count} leg, {single.total_tilt_deg:.6f} deg")
print(f"closed form arccos(1/sqrt(3)) = {exact:.6f} deg")
assert abs(single.total_tilt_deg - exact) < 1e-9
print()
print(single.describe())

The route reports the band to follow, the angle, and the intermediate zone axes it passes — the
landmarks that tell an operator the tilt is tracking rather than drifting. Now cap the leg length
and watch what changes and what does not.

In [ ]:
print(f"{'max leg':>8} {'hops':>5} {'total tilt':>11} {'direct':>8}   legs")
for budget in (60.0, 30.0, 20.0, 12.0):
    route = CUBIC_MAP.route_to([0, 0, 1], [1, 1, 1], max_leg_deg=budget)
    if not route.reachable:
        print(f"{budget:>8.0f} {'-':>5} {'unreachable':>11}")
        continue
    steps = " -> ".join(
        [str(route.legs[0].start_indices)] + [str(leg.end_indices) for leg in route.legs]
    )
    print(f"{budget:>8.0f} {route.hop_count:>5} {route.total_tilt_deg:>11.4f}"
          f" {route.direct_tilt_deg:>8.4f}   {steps}")

The total travel does not change. Splitting an arc at points that lie *on* the same great circle
cannot lengthen it, so multi-hop routing along one band is free in travel — and section 5 shows what
it buys.

## 5. Why several short hops beat one long one, exactly

An operator at a zone axis knows the beam direction but not the *azimuthal* orientation of the
crystal about it: the rotation between the diffraction pattern and the stage axes is calibrated
only to a degree or two. Call that uncertainty $\delta\varphi$.

Suppose the intended tilt is by $\theta$ about an axis $\mathbf{n}$ perpendicular to the current
zone axis $\mathbf{u}_0$. Because of $\delta\varphi$ the axis actually used is
$\mathbf{n}' = R(\mathbf{u}_0, \delta\varphi)\,\mathbf{n}$, and

$$R(\mathbf{n}', \theta) = R(\mathbf{u}_0, \delta\varphi)\,R(\mathbf{n}, \theta)\,
R(\mathbf{u}_0, \delta\varphi)^{-1}.$$

Since $R(\mathbf{u}_0, \cdot)$ fixes $\mathbf{u}_0$, applying this to $\mathbf{u}_0$ gives
$R(\mathbf{u}_0, \delta\varphi)\,\mathbf{u}_1$: the achieved direction is the *intended* one rotated
about the starting axis by the calibration error. So the angular miss is exactly

$$\boxed{\;\Delta = 2\arcsin\!\left(\sin\frac{\delta\varphi}{2}\,\sin\theta\right)
       \;\approx\; \delta\varphi\,\sin\theta\;}$$

— proportional to $\sin$ of the *hop length*. Split the excursion into $n$ hops and re-index at each
one, and each hop contributes $\delta\varphi \sin(\theta/n)$ with an independent error, so the
expected total goes as $\sqrt{n}\,\sin(\theta/n)$, which decreases with $n$.

In [ ]:
def rotation_about(axis, angle_deg):
    unit = np.asarray(axis, dtype=np.float64)
    unit = unit / np.linalg.norm(unit)
    angle = np.radians(angle_deg)
    skew = np.array([[0.0, -unit[2], unit[1]],
                     [unit[2], 0.0, -unit[0]],
                     [-unit[1], unit[0], 0.0]])
    return np.eye(3) + np.sin(angle) * skew + (1.0 - np.cos(angle)) * (skew @ skew)

start_axis = np.array([0.0, 0.0, 1.0])
target_axis = np.array([1.0, 1.0, 1.0]) / np.sqrt(3.0)
span = np.degrees(np.arccos(float(start_axis @ target_axis)))

print(f"one hop of {span:.4f} deg\n")
print(f"{'dphi (deg)':>11} {'simulated miss':>16} {'closed form':>13} {'dphi sin(theta)':>17}")
for offset_deg in (0.5, 1.0, 2.0, 5.0):
    achieved = rotation_about(start_axis, offset_deg) @ target_axis
    simulated = np.degrees(np.arccos(np.clip(float(achieved @ target_axis), -1.0, 1.0)))
    closed = np.degrees(
        2.0 * np.arcsin(np.sin(np.radians(offset_deg) / 2.0) * np.sin(np.radians(span)))
    )
    print(f"{offset_deg:>11.1f} {simulated:>16.5f} {closed:>13.5f}"
          f" {offset_deg * np.sin(np.radians(span)):>17.5f}")
    assert abs(simulated - closed) < 1e-9

In [ ]:
def expected_miss_deg(offset_deg, span_deg, hops):
    leg = span_deg / hops
    per_leg = np.degrees(
        2.0 * np.arcsin(np.sin(np.radians(offset_deg) / 2.0) * np.sin(np.radians(leg)))
    )
    # Independent errors at each re-indexing step add in quadrature.
    return float(per_leg * np.sqrt(hops))

offsets = (1.0, 2.0, 5.0)
hop_counts = np.arange(1, 9)
fig, ax = plt.subplots(figsize=(7.2, 4.2))
print(f"{'hops':>5} {'leg (deg)':>10} " + " ".join(f"{f'miss @ {o} deg':>15}" for o in offsets))
for hops in hop_counts:
    row = [expected_miss_deg(offset, span, int(hops)) for offset in offsets]
    print(f"{hops:>5} {span / hops:>10.3f} " + " ".join(f"{value:>15.4f}" for value in row))
for offset, colour in zip(offsets, ("#1f77b4", "#ff7f0e", "#d62728")):
    ax.plot(hop_counts, [expected_miss_deg(offset, span, int(n)) for n in hop_counts],
            "o-", ms=4, color=colour, label=rf"$\delta\varphi = {offset:g}^\circ$")
ax.set_xlabel("hops the excursion is split into")
ax.set_ylabel("expected angular miss at the target (deg)")
ax.set_title(r"$[001] \rightarrow [111]$: the cost of one long tilt")
ax.legend(fontsize=8)
fig.tight_layout()

> **The awe note.** For a $2^\circ$ azimuthal calibration error, one $54.7^\circ$ hop lands
> $1.63^\circ$ from $[111]$ — comparable with the width of a *band*, so the operator arrives with the
> pattern visibly off and no clear indication of which way to correct. Three hops along the same band
> land $1.08^\circ$ away, six hops $0.78^\circ$, for exactly the same total stage travel. The
> improvement is not from better mechanics or a better calculation; it comes from *measuring again
> partway*, which converts an open-loop tilt into a closed loop. This is the same argument that makes
> dead reckoning worse than taking a fix, and it is why `DEFAULT_ROUTE_MAX_LEG_DEG` is 30 rather
> than unbounded.

The curve flattens, and the flattening is informative: past about four hops the $\sqrt{n}$
accumulation of independent errors starts to cancel the benefit of shorter legs, so there is an
optimum rather than a monotone gain. Splitting a $55^\circ$ excursion into twenty steps would be
work for nothing.

## 6. The hexagonal map

Everything above was stated for a lattice, not for cubic symmetry, so the same construction applies
unchanged to zirconium. Two things do change in the *reporting*, and both matter.

Directions and planes are named in four-index **Miller–Bravais** notation, $[uvtw]$ and $(hkil)$
with $t = -(u+v)$ and $i = -(h+k)$. The three-index form is not wrong, it is unreadable: the six
members of $\langle 2\bar{1}\bar{1}0 \rangle$ are $[100]$, $[010]$, $[\bar{1}10]$, ... which do not
look like one family, whereas in four indices they are permutations. PyTex switches notation on the
crystal system, so a hexagonal map names itself hexagonally.

And the $c/a$ ratio enters the *positions*. In a cubic map the pole positions are fixed by symmetry
alone; in a hexagonal one, where $\langle 2\bar{1}\bar{1}3 \rangle$ sits depends on $c/a$, which is
1.593 for zirconium against the ideal close-packed 1.633.

In [ ]:
HEX_MAP = compute_kikuchi_map(
    ZIRCONIUM, beam_energy_kev=BEAM_KEV, max_index=3, zone_axis_max_index=3,
    min_zone_axis_order=3,
)
print(HEX_MAP.describe())
print()
ratio = ZIRCONIUM.lattice.c / ZIRCONIUM.lattice.a
print(f"zirconium c/a = {ratio:.4f}   ideal close packed = {np.sqrt(8.0 / 3.0):.4f}")
for axis in HEX_MAP.zone_axes[:6]:
    print("  " + axis.describe())

In [ ]:
hex_route = HEX_MAP.route_to([0, 0, 1], [1, 0, 0], max_leg_deg=30.0)
print(hex_route.describe())
assert abs(hex_route.direct_tilt_deg - 90.0) < 1e-9

The basal and prismatic zone axes are exactly $90^\circ$ apart — the $c$ axis is perpendicular to
every direction in the basal plane whatever $c/a$ — so the total travel is a right angle, split into
four legs along the single $(01\bar{1}0)$ band. That is the classical hcp tilt experiment, and the
map has produced it from the lattice.

In [ ]:
figure = plot_kikuchi_map(
    HEX_MAP,
    route=hex_route,
    max_bands=16,
    width_scale=8.0,
    min_label_order=8,
    max_labels=10,
    title="Zirconium Kikuchi map from [0001] to a prismatic axis (widths x8)",
)

The six-fold symmetry of the band network is the point group made visible, and it is worth stating
what that means: the *map* has the symmetry of the crystal because the band set is a union of
symmetry families and the projection is centred on a symmetry axis. Centre the same map on a general
direction and the picture loses its symmetry while describing the same crystal — the symmetry is in
the network, and the projection merely displays it.

## 7. The two maps side by side

The same construction, the same code path, two crystal systems. What differs is the symmetry of the
network and the angles between the landmarks.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.0, 6.2))
plot_kikuchi_map(CUBIC_MAP, max_bands=14, width_scale=8.0, min_label_order=10, max_labels=7,
                 title="nickel, cubic, centred on [001]", ax=axes[0])
plot_kikuchi_map(HEX_MAP, max_bands=14, width_scale=8.0, min_label_order=10, max_labels=7,
                 title="zirconium, hexagonal, centred on [0001]", ax=axes[1])
fig.tight_layout()

print(f"{'phase':<18} {'bands':>6} {'zone axes':>10} {'highest order':>14} {'that axis':>12}")
for label, kikuchi_map in (("nickel-fcc", CUBIC_MAP), ("zirconium-hcp", HEX_MAP)):
    top = kikuchi_map.zone_axes[0]
    print(f"{label:<18} {kikuchi_map.band_count:>6} {kikuchi_map.zone_axis_count:>10}"
          f" {top.order:>14} {str(top.indices):>12}")

## 8. Failure modes, deliberately triggered

**(a) Asking for an axis the map does not carry.** The map is built to a finite index bound, so a
high-index target is simply absent, and the error says which knob to turn rather than failing
opaquely.

In [ ]:
try:
    plan_kikuchi_route(CUBIC_MAP, [0, 0, 1], [7, 9, 11])
except ValueError as error:
    print("caught:", error)

**(b) A leg budget nothing can satisfy.** An unreachable target is reported, not raised: the
question was reasonable and the honest answer is "not with that budget".

In [ ]:
impossible = CUBIC_MAP.route_to([0, 0, 1], [1, 1, 1], max_leg_deg=1.0)
print("reachable:", impossible.reachable)
print(impossible.describe())
print("\nJSON contract still describes the failure:",
      {key: impossible.to_json_dict()[key] for key in ("reachable", "hop_count", "target")})

**(c) Reading the fattest band as the strongest.** Section 1 measured the two orderings; here is
what happens if you conflate them and go looking for the widest band on the screen.

In [ ]:
by_width = sorted(CUBIC_MAP.bands, key=lambda band: -band.angular_width_deg)[:3]
by_intensity = CUBIC_MAP.bands[:3]
print(f"{'ranked by width':<34} {'ranked by intensity':<34}")
for wide, strong in zip(by_width, by_intensity):
    print(f"{str(wide.indices) + f'  I={wide.relative_intensity:.4f}':<34} "
          f"{str(strong.indices) + f'  w={strong.angular_width_deg:.3f} deg':<34}")
print("\nThe widest bands carry a few percent of the strongest band's intensity, so the band an")
print("operator can actually see is never the one this ranking puts first.")

**(d) Forgetting that $[uvw]$ and $[\bar{u}\bar{v}\bar{w}]$ are the same zone axis.** A zone axis is
a *line*: the beam traverses the crystal in one direction, and the pattern from $[uvw]$ and
$[\bar u\bar v\bar w]$ is the same. So a route asked for $[2\bar1\bar10]$ may legitimately arrive at
$[\bar2110]$, and a route's reported indices carry the sense it actually travels rather than the
map's own canonical representative.

In [ ]:
requested = [1, 0, 0]
route = HEX_MAP.route_to([0, 0, 1], requested, max_leg_deg=30.0)
canonical = HEX_MAP.zone_axis_for_direction(requested)
print("requested 3-index axis      :", tuple(requested))
print("map's canonical direction   :", np.asarray(canonical.direction_map))
print("route's target indices      :", route.target_indices)
print("route's target direction    :", np.asarray(route.target_direction))

alignment = float(np.asarray(canonical.direction_map) @ np.asarray(route.target_direction))
print(f"\ndot product of the two directions: {alignment:+.1f}")
print("Exactly -1: antiparallel, therefore the same line, therefore the same zone axis --")
print("the beam traverses the crystal one way or the other and the pattern is identical.")
print("A test that compared the index triples for equality would report a routing failure.")
assert abs(alignment + 1.0) < 1e-12

## 9. What this implementation does not do

- **Kinematic intensities.** The $|F_{\mathbf{g}}|^2$ ordering is right for deciding which bands are
  prominent, but the map does not predict the excess–deficiency asymmetry across a band, dynamical
  contrast, or higher-order Laue zone effects. Tutorial 29 covers what a dynamical calculation adds.
- **No specimen, no instrument.** The map is a map of the *crystal*: no foil thickness, no
  absorption, no detector. It says which way to turn, not what the screen will look like.
- **One band per plane trace by default.** All orders of a reflection share a centre line, so higher
  orders are folded onto the lowest allowed one. `include_higher_orders=True` unfolds them, at the
  cost of coincident lines and inflated zone-axis orders.
- **Angles between crystal directions, not stage angles.** A route's tilt is the angle the *crystal*
  must turn through. Converting it into $\alpha$ and $\beta$ needs a calibrated holder, and checking
  it against the mechanical envelope needs `pytex.tem.navigation` and `pytex.tem.path` — which is
  what tutorial 24 does, and what `KikuchiRoute.describe()` says explicitly rather than leaving
  implicit.
- **Routing minimizes travel, not risk.** The shortest band-followable route is not necessarily the
  most robust one: section 5's argument would sometimes prefer a longer path through more
  intersections. The search does not model that trade.

## 10. What to take away

- **Width measures $d$ inversely; intensity decides visibility.** $2\theta_B = 2\arcsin(\lambda/2d)$
  is under a degree for any spacing worth measuring, which is precisely why the network is legible.
- **The atlas must be stereographic.** The gnomonic radius at the map boundary is infinite. And
  stereographic is conformal, so the angles read off the map are the angles the stage turns through.
- **The zone law is integer arithmetic.** Zone axes and their orders are exact combinatorics on the
  reflection list, and the order — not the indices — tells you whether you will recognize the axis
  when you get there.
- **Follow the band; it is the geodesic.** Both are defined by the plane the two axes span, so the
  path an experienced operator takes by eye is the shortest one available.
- **Split long excursions.** The miss goes as $\delta\varphi\sin\theta$ per hop, so re-indexing
  partway is worth about a factor of 1.5 on a $55^\circ$ excursion at no cost in travel — and the
  benefit saturates around four hops.
- **A zone axis is a line.** $\pm[uvw]$ are the same axis, and any code that compares axes by index
  equality will eventually report a false failure.

### Further reading

- Tutorial 24, *TEM tilt navigation* — converting a route into stage $\alpha$ and $\beta$ and
  validating it against the holder envelope.
- Tutorial 27, *TEM diffraction pattern indexing as a round trip* — determining the orientation you
  are starting from, and the ambiguities a single pattern leaves.
- Tutorial 29, *Dynamical CBED and point groups* — what the intensities in this map do not model.
- Tutorial 10, *Plotting semantic primitives* — stereographic versus equal-area projection in the
  general case.
- `docs/tex/algorithms/stereographic_kikuchi_maps.tex` — the derivations, including the tilt-error
  identity of section 5.
- N. K. Levine, W. L. Bell and G. Thomas, *J. Appl. Phys.* **37** (1966) 2141 — the original
  montaged Kikuchi maps.
- J. W. Edington, *Practical Electron Microscopy in Materials Science*, Monograph 2 (Philips, 1975)
  — Kikuchi maps as an operating technique.
- D. B. Williams and C. B. Carter, *Transmission Electron Microscopy*, 2nd ed. (Springer, 2009),
  Ch. 19 — Kikuchi lines, their geometry, and their use for tilting.